# 03 - Train the two deep models (Part B-1)

**DeepLabV3+** (CNN family) and **SegFormer-B3** (transformer family) are trained with the *identical* pipeline: same images, same augmentation, same image size (512), same optimizer (AdamW), same schedule (poly), same epochs. That is what makes the comparison fair, and the brief explicitly requires it.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q segmentation-models-pytorch transformers optuna thop joblib
print('installs done')

installs done


In [4]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')

GPU: NONE - Runtime > Change runtime type > T4 GPU


In [5]:
frames = data.load_frames()
print('frames loaded:', {s: len(f) for s, f in frames.items()})

masks index: 10000 rows | splits: {'train': 8000, 'val': 1000, 'test': 1000} | missing files: 0
                                                                            image                                                          mask  split
0  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000307967.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000307967.png  train
1  /content/drive/MyDrive/coco_seg/fiftyone/coco-2017/train/data/000000154345.jpg  /content/drive/MyDrive/coco_seg/masks/train/000000154345.png  train
frames loaded: {'train': 8000, 'val': 1000, 'test': 1000}


In [ ]:
import shutil, os
SRC = '/content/drive/MyDrive/coco_seg'
DST = '/content/local_ds'
os.makedirs(DST, exist_ok=True)
for sub in ['fiftyone', 'masks']:
    s, d = os.path.join(SRC, sub), os.path.join(DST, sub)
    if not os.path.exists(d):
        print('copying', sub, '...')
        shutil.copytree(s, d, ignore=shutil.ignore_patterns('*.zip'))
for split in frames:
    for col in frames[split].columns:
        if frames[split][col].dtype == object:
            frames[split][col] = frames[split][col].str.replace(SRC, DST, regex=False)
print('copy done, first image local:', os.path.exists(frames['train']['image'].iloc[0]))

copying fiftyone ...
copying masks ...
copy done, first image local: True


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

### Train DeepLabV3+ (base loss: cross-entropy)

In [ ]:
name = 'deeplabv3plus'
train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True)
val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
model = models.build_model(name)
history = engine.train(model, f'{name}_main_ce', train_loader, val_loader,
                       losses.build_loss('ce', C.NUM_CLASSES), epochs=C.EPOCHS_MAIN, seed=C.SEEDS[0])
val_metrics = engine.evaluate(model, val_loader)
val_metrics['seconds_per_epoch'] = float(np.mean([h['epoch_seconds'] for h in history]))
report_utils.save_result(f'train_main_{name}', {'history': history, 'val': val_metrics})
__import__('torch').save(model.state_dict(), os.path.join(C.CKPT_DIR, f'{name}_main_ce_final.pt'))
print(name, 'val mIoU:', round(val_metrics['miou'], 4))

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[resume] deeplabv3plus_main_ce: continuing at epoch 6/12
deeplabv3plus_main_ce epoch 6/12  loss 0.1621  val mIoU 0.4877  (386s)
deeplabv3plus_main_ce epoch 7/12  loss 0.1445  val mIoU 0.5355  (372s)
deeplabv3plus_main_ce epoch 8/12  loss 0.1281  val mIoU 0.5563  (369s)
deeplabv3plus_main_ce epoch 9/12  loss 0.1148  val mIoU 0.5676  (374s)
deeplabv3plus_main_ce epoch 10/12  loss 0.1051  val mIoU 0.5799  (382s)
deeplabv3plus_main_ce epoch 11/12  loss 0.1081  val mIoU 0.5864  (390s)
deeplabv3plus_main_ce epoch 12/12  loss 0.0998  val mIoU 0.5869  (385s)
deeplabv3plus val mIoU: 0.5869


### Train SegFormer-B3 (same pipeline, same base loss)

In [ ]:
name = 'segformer_b3'
train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True)
val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
model = models.build_model(name)
history = engine.train(model, f'{name}_main_ce', train_loader, val_loader,
                       losses.build_loss('ce', C.NUM_CLASSES), epochs=C.EPOCHS_MAIN, seed=C.SEEDS[0])
val_metrics = engine.evaluate(model, val_loader)
val_metrics['seconds_per_epoch'] = float(np.mean([h['epoch_seconds'] for h in history]))
report_utils.save_result(f'train_main_{name}', {'history': history, 'val': val_metrics})
__import__('torch').save(model.state_dict(), os.path.join(C.CKPT_DIR, f'{name}_main_ce_final.pt'))
print(name, 'val mIoU:', round(val_metrics['miou'], 4))

the training loss falling and the val mIoU climbing each epoch. If val mIoU stays near 0.05 after several epochs, something is wrong